# Stage 15: fold-safe inference package

Colabで最終packageを作ります。testと同じIDのtrain坑井を学習から除外したOOF対応モデルだけを収録します。提出はまだ作りません。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
data_dir = drive_root / 'data'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
assert (data_dir / 'test').is_dir() and (data_dir / 'sample_submission.csv').is_file()
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

## 既存artifactを検証してpackageを構築

Stage 11/12B/12C/14/14Bを再学習せず利用します。surfaceモデルのみ小さな表データでfold別に再学習します。

In [ ]:
runs = {
    'stage11': artifact_dir / 'stage11_multicut_delta_u_full_v001',
    'stage12b': artifact_dir / 'stage12b_learned_emission_tcn_full_v001',
    'stage12c': artifact_dir / 'stage12c_spatial_kbest_lattice_full_v001',
    'stage14': artifact_dir / 'stage14_crossfit_emission_residual_full_v001',
    'stage14b': artifact_dir / 'stage14b_extended_residual_gate_full_v001',
}
for name, path in runs.items():
    assert path.is_dir(), f'Missing {name}: {path}'
summary = json.loads((runs['stage14b'] / 'gate_summary.json').read_text())
assert summary['promoted_to_full_residual_training'] is True
assert summary['robust_extended_generic_spec'] == 'generic_w080_cap16'
print('Stage 14B promotion verified')

In [ ]:
RUN_ID = 'stage15_fold_safe_package_v001'
run_dir = artifact_dir / RUN_ID
command = [
    'uv', 'run', 'rogii-stage15-package',
    '--config', 'configs/experiment/stage15_fold_safe_package.yaml',
    '--stage11-run', str(runs['stage11']), '--stage12b-run', str(runs['stage12b']),
    '--stage12c-run', str(runs['stage12c']), '--stage14-run', str(runs['stage14']),
    '--stage14b-run', str(runs['stage14b']), '--data-dir', str(data_dir),
    '--artifact-dir', str(artifact_dir), '--run-id', RUN_ID,
]
if not (run_dir / 'package_summary.json').is_file():
    subprocess.run(command, cwd=repo_dir, check=True)
else:
    print('Reusing completed package:', run_dir)
json.loads((run_dir / 'package_summary.json').read_text())

## 次に行うこと

Driveの `artifacts/stage15_fold_safe_package_v001/package` フォルダをKaggle Datasetとしてアップロードします。zipは保管・ダウンロード用です。Kaggle Notebookでは次の340 notebookをコピーして実行します。